# Week 7 Assignment
# Document Question Answering System using RAG

## Name: Rishabh Sah

### Objective
To build a Retrieval-Augmented Generation (RAG) system that answers questions from custom documents.

### Workflow
1. Load PDF document
2. Split text into chunks
3. Create embeddings
4. Store embeddings in FAISS
5. Retrieve relevant chunks
6. Generate answers using Gemini

In [21]:
!pip install -q langchain
!pip install -q langchain-community
!pip install -q langchain-huggingface
!pip install -q faiss-cpu
!pip install -q pypdf
!pip install -q sentence-transformers
!pip install -q google-generativeai

In [22]:
import os

from google.colab import files

from langchain_community.document_loaders import PyPDFLoader

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_huggingface import HuggingFaceEmbeddings

from langchain_community.vectorstores import FAISS

import google.generativeai as genai


In [23]:
uploaded = files.upload()

Saving Rishabh Sah_PCE23AD042 (2).pdf to Rishabh Sah_PCE23AD042 (2) (1).pdf


In [24]:
loader = PyPDFLoader("Rishabh Sah_PCE23AD042 (2).pdf")

documents = loader.load()

print("Total Pages Loaded:", len(documents))

Total Pages Loaded: 1


## Text Chunking

In [25]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(documents)

print("Total Chunks:", len(chunks))

Total Chunks: 4


In [26]:
for i, chunk in enumerate(chunks[:3]):
    print(f"\nChunk {i+1}\n")
    print(chunk.page_content)
    print("-"*80)


Chunk 1

RISHABH SAH
Software Developer
7981132525 — rishabhsah143@gmail.com
linkedin — github
SUMMARY
Aspiring Software Developer with strong foundations in programming, OOPs, and core data structures. Skilled in
Python and C++ with experience in building applications and working with MySQL.
COURSEWORK / SKILLS
•Data Structures & Algorithms
•Artificial Intelligence
•Machine Learning
•DataBase Management System (DBMS)
•Web Development
•OOPS Concept
•Networking System
•Operating Systems
TECHNICAL SKILLS
--------------------------------------------------------------------------------

Chunk 2

•Operating Systems
TECHNICAL SKILLS
Languages:C++, Python, HTML, CSS
Database:MySQL
T ools:Git, GitHub, VS Code
Concepts:OOPs, API Integration, Problem Solving
PROJECTS
NeoNest – AI-Enabled Healthcare Platform
HTML, CSS, Python /github/external-link-alt
•Developed responsive web application for healthcare services
•Implemented user workflows and onboarding
•Designed frontend with responsive layout

## Create Embeddings

In [27]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding Model Loaded Successfully")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding Model Loaded Successfully


## Create Vector Database using FAISS

In [28]:
vector_db = FAISS.from_documents(
    chunks,
    embeddings
)

print("FAISS Vector Store Created Successfully")

FAISS Vector Store Created Successfully


## Configure Gemini API

In [29]:
genai.configure(api_key="YOUR_API_KEY")

model = genai.GenerativeModel("gemini-2.0-flash")

print("Gemini Configured Successfully")

Gemini Configured Successfully


## Retrieval and Question Answering Function

In [30]:
!pip install -q transformers torch accelerate

In [31]:
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="google/flan-t5-base"
)

print("Local LLM Loaded Successfully")

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
[transformers] The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'Cohere2MoeForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM',

Local LLM Loaded Successfully


In [32]:
def ask_question(query):

    retriever = vector_db.as_retriever(search_kwargs={"k": 3})

    docs = retriever.invoke(query)

    print("\nRetrieved Context:\n")
    print("=" * 100)

    context = ""

    for i, doc in enumerate(docs):
        print(f"\nChunk {i+1}:\n")
        print(doc.page_content)
        print("-" * 100)

        context += doc.page_content + "\n"

    prompt = f"""
Answer the question only from the provided context.

Context:
{context}

Question:
{query}

If the answer is not present in the context, say:
"Answer not found in document."

Answer:
"""

    print("\nGenerating Answer...\n")

    try:
        response = generator(
            prompt,
            max_new_tokens=100,
            do_sample=False
        )

        print("\nGenerated Answer:\n")
        print("=" * 100)

        print(response[0]['generated_text'])

    except Exception as e:
        print("Error while generating answer:")
        print(e)

## Test Questions

In [33]:
ask_question("What projects are mentioned in the document?")

[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Retrieved Context:


Chunk 1:

•Operating Systems
TECHNICAL SKILLS
Languages:C++, Python, HTML, CSS
Database:MySQL
T ools:Git, GitHub, VS Code
Concepts:OOPs, API Integration, Problem Solving
PROJECTS
NeoNest – AI-Enabled Healthcare Platform
HTML, CSS, Python /github/external-link-alt
•Developed responsive web application for healthcare services
•Implemented user workflows and onboarding
•Designed frontend with responsive layout
LLM Hallucination Auditor – AI V erification System
Python, NLP, NumPy, Pandas
----------------------------------------------------------------------------------------------------

Chunk 2:

Python, NLP, NumPy, Pandas
•Built system to validate AI-generated responses
•Implemented scoring logic and modular design
•Optimized data processing workflows
INTERNSHIP
CREA TIX Pvt. Ltd. 2025
Summer Intern
•Developed AI chatbot and implemented TF-IDF search
•Integrated Google Gemini API
EDUCATION
B.T ech – Artificial Intelligence & Data ScienceCGPA: 8
Poornima College of 

In [34]:
ask_question("What technical skills are mentioned?")

[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Retrieved Context:


Chunk 1:

•Operating Systems
TECHNICAL SKILLS
Languages:C++, Python, HTML, CSS
Database:MySQL
T ools:Git, GitHub, VS Code
Concepts:OOPs, API Integration, Problem Solving
PROJECTS
NeoNest – AI-Enabled Healthcare Platform
HTML, CSS, Python /github/external-link-alt
•Developed responsive web application for healthcare services
•Implemented user workflows and onboarding
•Designed frontend with responsive layout
LLM Hallucination Auditor – AI V erification System
Python, NLP, NumPy, Pandas
----------------------------------------------------------------------------------------------------

Chunk 2:

Python, NLP, NumPy, Pandas
•Built system to validate AI-generated responses
•Implemented scoring logic and modular design
•Optimized data processing workflows
INTERNSHIP
CREA TIX Pvt. Ltd. 2025
Summer Intern
•Developed AI chatbot and implemented TF-IDF search
•Integrated Google Gemini API
EDUCATION
B.T ech – Artificial Intelligence & Data ScienceCGPA: 8
Poornima College of 

In [35]:
ask_question("What is the educational qualification?")

[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Retrieved Context:


Chunk 1:

Higher Secondary Education 83.3%
ACHIEVEMENTS
•Top 8 Finalist – Smart Ideathon 2K25 (2300+ teams)/external-link-alt
CERTIFICATIONS
•Google Cloud Skill Badge – AI Applications with Gemini & Imagen
•Participated in Code Slayer (NIT Delhi), HackWithMAIT, ISRO Hackathon/external-link-alt
----------------------------------------------------------------------------------------------------

Chunk 2:

Python, NLP, NumPy, Pandas
•Built system to validate AI-generated responses
•Implemented scoring logic and modular design
•Optimized data processing workflows
INTERNSHIP
CREA TIX Pvt. Ltd. 2025
Summer Intern
•Developed AI chatbot and implemented TF-IDF search
•Integrated Google Gemini API
EDUCATION
B.T ech – Artificial Intelligence & Data ScienceCGPA: 8
Poornima College of Engineering, Jaipur
Higher Secondary Education 83.3%
ACHIEVEMENTS
----------------------------------------------------------------------------------------------------

Chunk 3:

RISHABH SAH
Softw

In [36]:
ask_question("Summarize the document.")

[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Retrieved Context:


Chunk 1:

•Operating Systems
TECHNICAL SKILLS
Languages:C++, Python, HTML, CSS
Database:MySQL
T ools:Git, GitHub, VS Code
Concepts:OOPs, API Integration, Problem Solving
PROJECTS
NeoNest – AI-Enabled Healthcare Platform
HTML, CSS, Python /github/external-link-alt
•Developed responsive web application for healthcare services
•Implemented user workflows and onboarding
•Designed frontend with responsive layout
LLM Hallucination Auditor – AI V erification System
Python, NLP, NumPy, Pandas
----------------------------------------------------------------------------------------------------

Chunk 2:

Python, NLP, NumPy, Pandas
•Built system to validate AI-generated responses
•Implemented scoring logic and modular design
•Optimized data processing workflows
INTERNSHIP
CREA TIX Pvt. Ltd. 2025
Summer Intern
•Developed AI chatbot and implemented TF-IDF search
•Integrated Google Gemini API
EDUCATION
B.T ech – Artificial Intelligence & Data ScienceCGPA: 8
Poornima College of 

## System Metrics Report

In [37]:
print("===== SYSTEM METRICS =====")

print("Total Documents:", len(documents))
print("Total Chunks:", len(chunks))
print("Chunk Size:", 500)
print("Chunk Overlap:", 50)

print("Embedding Model: all-MiniLM-L6-v2")
print("Embedding Dimension: 384")

print("Vector Database: FAISS")
print("Retriever Top-K:", 3)

===== SYSTEM METRICS =====
Total Documents: 1
Total Chunks: 4
Chunk Size: 500
Chunk Overlap: 50
Embedding Model: all-MiniLM-L6-v2
Embedding Dimension: 384
Vector Database: FAISS
Retriever Top-K: 3


# Experiment and Observations

1. Chunk size of 500 with overlap of 50 provided relevant context.
2. Smaller chunks improved precision but sometimes lost context.
3. FAISS enabled fast similarity search.
4. The RAG system generated grounded answers based on retrieved chunks.
5. Using custom PDFs allows answering questions on private documents.

# Conclusion

This project successfully implemented a Retrieval-Augmented Generation (RAG) system for document question answering.

The system loads custom PDF documents, converts them into embeddings, stores them in a FAISS vector database, retrieves relevant information, and generates context-aware answers using Gemini.

The approach improves factual accuracy and enables question answering over private documents.